In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import ast
import pandas as pd
from tqdm import tqdm
import numpy as np
from github_helper import from_github

In [4]:
df_votes = pd.DataFrame()
periods = [65, 66, 67, 68, 69, 70, 71]
for period in periods:
    df_period = pd.read_csv(from_github(f"/voting-data/votes_enriched_p{period}.csv"))
    print(f"Found {df_period["vote_id"].nunique()} votes by {df_period['aktørid'].nunique()} actors ({df_period["vote_id"].nunique() / df_period['aktørid'].nunique():.2f} avg.) on {df_period['afstemning_id'].nunique()} voting sessions in period {period}")
    df_votes = pd.concat([df_votes, df_period])

df_votes['all_topics'] = df_votes["all_topics"].apply(ast.literal_eval)
df_votes.head(10)


Found 28529 votes by 107 actors (266.63 avg.) on 294 voting sessions in period 65
Found 130072 votes by 134 actors (970.69 avg.) on 1269 voting sessions in period 66
Found 268954 votes by 213 actors (1262.69 avg.) on 1778 voting sessions in period 67
Found 249768 votes by 198 actors (1261.45 avg.) on 1727 voting sessions in period 68
Found 284754 votes by 201 actors (1416.69 avg.) on 2017 voting sessions in period 69
Found 229806 votes by 175 actors (1313.18 avg.) on 1688 voting sessions in period 70
Found 161991 votes by 173 actors (936.36 avg.) on 1302 voting sessions in period 71


,vote_id,vote_typeid,afstemning_id,aktørid,afstemning_nummer,afstemning_vedtaget,Period,primary_topic,all_topics,møde_dato,møde_year_month,politician,party
0,1484427,3,5986,235,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Jeppe Kofod,Socialdemokratiet
1,1484403,2,5986,1461,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Rikke Hvilshøj,Venstre
2,1484402,3,5986,1833,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Høgni Hoydal,Tjóðveldi
3,1484400,3,5986,364,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Birthe Rønn Hornbech,Venstre
4,1484398,3,5986,1883,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Anne Grete Holmsgaard (udpeget af SF),Socialistisk Folkeparti
5,1484396,1,5986,1444,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Poul Henrik Hedeboe,Socialistisk Folkeparti
6,1484404,3,5986,678,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Klaus Hækkerup (udpeget af S),Socialdemokratiet
7,1484395,3,5986,1891,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Thomas Adelskov,Socialdemokratiet
8,1484392,3,5986,164,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Hans Andersen,Venstre
9,1484390,2,5986,126,2,False,65,klima_miljø,"[klima_miljø, finans_budget, forsvar_sikkerhed...",2004-10-07 10:00:00,2004-10,Kim Andersen (udpeget af V),Venstre


In [ ]:

# df_votes['all_topics'] = df_votes['all_topics'].apply(
#     lambda x: [x] if isinstance(x, str) else x
# )

# df_votes.dtypes
# df_votes['all_topics'].head(20).apply(type)


0     <class 'list'>
1     <class 'list'>
2     <class 'list'>
3     <class 'list'>
4     <class 'list'>
5     <class 'list'>
6     <class 'list'>
7     <class 'list'>
8     <class 'list'>
9     <class 'list'>
10    <class 'list'>
11    <class 'list'>
12    <class 'list'>
13    <class 'list'>
14    <class 'list'>
15    <class 'list'>
16    <class 'list'>
17    <class 'list'>
18    <class 'list'>
19    <class 'list'>
Name: all_topics, dtype: object

In [41]:
def get_voting_percentages_for_party_on_voting_session(df):
    df_cast_votes = df[df["vote_typeid"] != 3]
    df_type_1 = df[df["vote_typeid"] == 1]
    df_type_2 = df[df["vote_typeid"] == 2]
    df_type_3 = df[df["vote_typeid"] == 3]
    df_type_4 = df[df["vote_typeid"] == 4]

    cols_to_group_by = ["afstemning_id", "party", "Period", "afstemning_vedtaget", "møde_year_month"]

    total_votes = (df.groupby(cols_to_group_by).size().rename("total_potential_votes"))
    cast_votes = (df_cast_votes.groupby(cols_to_group_by).size().rename("total_cast_votes"))

    votes_t1 = (df_type_1.groupby(cols_to_group_by).size().rename("votes_type_1"))
    votes_t2 = (df_type_2.groupby(cols_to_group_by).size().rename("votes_type_2"))
    votes_t3 = (df_type_3.groupby(cols_to_group_by).size().rename("votes_type_3"))
    votes_t4 = (df_type_4.groupby(cols_to_group_by).size().rename("votes_type_4"))

    voting_percentages = (
        total_votes.to_frame() #Create df for total_votes_shared
        .join([cast_votes, votes_t1, votes_t2, votes_t3, votes_t4], how = "left") #Left join as there are some people who shared votes but did not agree

        # .join(df_type_2, how = "left", on = ["voting_id", "party", "Period", "vedtaget", "Møde.dato"])
        .fillna(0)
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    voting_percentages["agree_percentage_of_cast_votes"] = voting_percentages["votes_type_1"] / voting_percentages["total_cast_votes"]
    voting_percentages["disagree_percentage_of_cast_votes"] = voting_percentages["votes_type_2"] / voting_percentages["total_cast_votes"]
    voting_percentages["abstain_percentage_of_cast_votes"] = voting_percentages["votes_type_4"] / voting_percentages["total_cast_votes"]
    voting_percentages["absent_percentage_of_potential_votes"] = voting_percentages["votes_type_3"] / voting_percentages["total_potential_votes"]

    voting_percentages.fillna(0, inplace = True)
    voting_percentages.to_csv("./voting-data/voting_session_party_percentages.csv", index = False)
    return voting_percentages

voting_per = get_voting_percentages_for_party_on_voting_session(df_votes)
voting_per.head()


,afstemning_id,party,Period,afstemning_vedtaget,møde_year_month,total_potential_votes,total_cast_votes,votes_type_1,votes_type_2,votes_type_3,votes_type_4,agree_percentage_of_cast_votes,disagree_percentage_of_cast_votes,abstain_percentage_of_cast_votes,absent_percentage_of_potential_votes
0,1,Dansk Folkeparti,68,True,2014-09,19,11.0,11.0,0.0,8.0,0.0,1.0,0.0,0.0,0.421053
1,1,Det Konservative Folkeparti,68,True,2014-09,8,5.0,5.0,0.0,3.0,0.0,1.0,0.0,0.0,0.375000
2,1,Enhedslisten,68,True,2014-09,12,7.0,7.0,0.0,5.0,0.0,1.0,0.0,0.0,0.416667
3,1,Inuit Ataqatigiit,68,True,2014-09,1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.000000
4,1,Liberal Alliance,68,True,2014-09,9,4.0,4.0,0.0,5.0,0.0,1.0,0.0,0.0,0.555556


In [21]:
def build_edges_for_period(filtered_df, period, period_col = "møde_year_month"):
    total_df_for_period = filtered_df[filtered_df[period_col] == period]
    df_period = total_df_for_period[['afstemning_id', 'vote_typeid', 'politician','party', period_col]] #, topic_col

    # #Now we find all pairs by joining the dataframe onto itself based on the 2 criteria.       
    pairs = df_period.merge(
        df_period
        , on = ["afstemning_id", period_col] # We do not group by the kind of vote they did, as we would rather do calculations on it seperately than create 3 df for each
        , suffixes = ("_source", "_target")
    )

    #Now we make sure we only have one observation per pair
    pairs = pairs[pairs['politician_source'] < pairs['politician_target']] #Only keep the ones where the politicians are different
    # 1) No duplicate pairs, so values are always different 
    # 2) No "reverse" pairs, as one has to be bigger than the other. If we had used != then there might have been (A,B) and (B,A)

    #Now we do something similar to ensure that we group the politician parties
    mask = pairs['party_source'] <= pairs['party_target']
    pairs['party_a'] = np.where(mask, pairs['party_source'], pairs['party_target'])
    pairs['party_b'] = np.where(mask, pairs['party_target'], pairs['party_source'])

    # #Now we have to count them.
    total_votes_shared_by_po = (
        pairs.groupby(["politician_source", "politician_target", "party_source", "party_target", period_col])
            .size() #Get the number of total votes shared 
            .rename("total_votes_shared")
    )

    total_votes_shared_by_pa = (
        pairs.groupby(['party_a', 'party_b', period_col])
        .size()
        .rename('total_votes_shared')
    )

    agreeing_pairs = pairs[pairs["vote_typeid_source"]==pairs["vote_typeid_target"]]
    agreed_votes_po = (
        agreeing_pairs.groupby(["politician_source", "politician_target"
                , "party_source", "party_target"
                , period_col]
            )
            .size()  
            .rename("total_votes_agreed")
    )

    agreed_votes_pa = (
        agreeing_pairs.groupby(["party_a", "party_b", period_col])
            .size()  
            .rename("total_votes_agreed")
    )

    #Calculate for the politician
    result_po = (
        total_votes_shared_by_po.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_po, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index() #Reset index releases the joined indexes so they become columns again
    )
    result_po['weight'] = result_po["total_votes_agreed"] / result_po["total_votes_shared"]

    #Calculate for the party
    result_pa = (
        total_votes_shared_by_pa.to_frame() #Create df for total_votes_shared
        .join(agreed_votes_pa, how = "left") #Left join as there are some people who shared votes but did not agree
        .fillna({'total_votes_agreed': 0})
        .reset_index()
    )
    result_pa['weight'] = result_pa["total_votes_agreed"] / result_pa["total_votes_shared"]

    return result_po, result_pa
all_periods = df_votes["møde_year_month"].unique()
period = all_periods[100]
filtered_df = df_votes[df_votes['vote_typeid'] != 3]

result_po, result_pa = build_edges_for_period(filtered_df, period, period_col="møde_year_month")


In [23]:
def build_edges_for_multiple_periods(df, topic, topic_col = "all_topics", period_col = "møde_year_month"):
    filtered_df = df[df['vote_typeid'] != 3]
    if topic != "general":
        df_exp = filtered_df.explode(topic_col) #Explode if we are calculating it for a specific topic
        filtered_df = df_exp[df_exp[topic_col] == topic] #Limit the dataframe to only the chosen topic

    # else:
    #     topic = "all_topics"

    all_periods = filtered_df[period_col].unique()
    all_results_for_politicians = []
    all_results_for_parties = []
    ##########################TURN OFF FOR PARALLEL
    print(f"Creating dataframe for {topic}, by {period_col}", end = " ")
    # for period in tqdm(all_periods): 
    for period in all_periods:
        result_po, result_pa = build_edges_for_period(filtered_df, period, period_col)
        all_results_for_politicians.append(result_po)
        all_results_for_parties.append(result_pa)

    all_votes_po = pd.concat(all_results_for_politicians, ignore_index = True)
    all_votes_pa = pd.concat(all_results_for_parties, ignore_index = True)

    #Rename for using correct names
    all_votes_po.rename(
        columns = {
            'politician_source' : 'source'
            ,'politician_target' : 'target'
            ,'party_source' : 'source_party'
            ,'party_target' : 'target_party'
        }
        , inplace = True
    )

    all_votes_pa.rename(
        columns = {
            'party_a' : 'source'
            ,'party_b' : 'target'
        }
        , inplace= True
    )

    #Add the topics
    all_votes_po['topic'] = topic
    all_votes_pa['topic'] = topic

    all_votes_po.to_csv(f"./edges/politician/{period_col}/politician_edges_{topic}_by_{period_col}.csv", index = False)
    all_votes_pa.to_csv(f"./edges/party/{period_col}/party_edges_{topic}_by_{period_col}.csv", index = False)

    # return all_votes_po, all_votes_pa

# all_votes_po, all_votes_pa = build_edges_for_multiple_periods(df_votes, topic = "bolig", topic_col="all_topics", period_col = "møde_year_month")
# build_edges_for_multiple_periods(df_votes, topic = "bolig", topic_col="all_topics", period_col = "møde_year_month")

In [25]:
all_topics = df_votes.explode("all_topics")['all_topics'].dropna().unique() #Drop na, because it would otherwise return an "na" column, which we don't want
topics_for_loop = [topic for topic in all_topics]
topics_for_loop.append("general") 

for topic in tqdm(topics_for_loop):
    # print(f"Creating dataframe for {topic}", end = " ")
    build_edges_for_multiple_periods(df_votes, topic = topic, topic_col="all_topics", period_col = 'møde_year_month')

  0%|          | 0/15 [00:00<?, ?it/s]

Creating dataframe for klima_miljø, by møde_year_month 

  7%|▋         | 1/15 [00:49<11:38, 49.91s/it]

Creating dataframe for finans_budget, by møde_year_month 

 13%|█▎        | 2/15 [02:05<14:07, 65.22s/it]

...

 87%|████████▋ | 13/15 [09:47<01:14, 37.30s/it]

Creating dataframe for immigration, by møde_year_month 

 93%|█████████▎| 14/15 [10:13<00:33, 33.73s/it]

Creating dataframe for general, by møde_year_month 

100%|██████████| 15/15 [12:34<00:00, 50.31s/it]


In [26]:
# # # # # # Not worth it, it's too heavy
# from joblib import Parallel, delayed
# all_topics = df_votes.explode("all_topics")['all_topics'].dropna().unique() #Drop na, because it would otherwise return an "na" column, which we don't want
# topics_for_loop = [topic for topic in all_topics]
# topics_for_loop.append("general") 
# topic_col = "all_topics"
# period_col = "Period"

# results = Parallel(n_jobs=-1)(delayed(build_edges_for_multiple_periods)(
#     df_votes
#     ,topic
#     ,topic_col
#     , period_col
# ) for topic in topics_for_loop)

In [27]:
#alright let's just make it one big dataframe yeah? Doing it on politician level is too much though
all_party_edges = pd.DataFrame()
# all_politician_edges = pd.DataFrame()
for topic in topics_for_loop:
    # politician_df = pd.read_csv(f"./edges/politician_edges_{topic}.csv")
    party_df = pd.read_csv(f"./edges/party/Period/party_edges_{topic}_by_Period.csv")

    # all_politician_edges = pd.concat([all_politician_edges, politician_df])
    all_party_edges = pd.concat([all_party_edges, party_df])

# all_politician_edges.to_csv(f"./edges/all_politician_edges.csv", index = False)
all_party_edges.to_csv(f"./edges/party/all_party_edges_by_Period.csv", index = False)